# Most-Cited Scholarly Works

Starter notebook for the Kaggle dataset in this folder. It loads `crossref_works.csv` with pandas and walks through quality checks and a first piece of analysis.

**Files:** `crossref_works.csv`, `data_dictionary.csv`.

## 1. Load the data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("crossref_works.csv")
df["published_year"] = pd.to_numeric(df["published_year"], errors="coerce")
df["citations"] = pd.to_numeric(df["citations"], errors="coerce")
print(df.shape)

## 2. Shape, types and missing values

In [ ]:
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")

missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0].round(1).to_frame("missing_%").head(15)

## 3. Numeric summary

In [ ]:
df.describe(include='number').T.round(2).head(20)

## 4. Citation distribution

Citations are extremely heavy-tailed — plot on a log scale.

In [ ]:
df["citations"].dropna().clip(lower=1).plot.hist(bins=60, log=True, figsize=(9, 5))

## 5. Open access vs citations

Compare the citation distribution for works with and without a registered licence.

In [ ]:
df.groupby("is_open_access")["citations"].describe().round(1)

## 6. Team size and impact

In [ ]:
(df.groupby("team_size_class")["citations"]
   .agg(["count", "median", "mean"]).round(1)
   .sort_values("median", ascending=False))

## 7. Leading journals by mean citations

In [ ]:
top = (df.groupby("journal")
         .agg(articles=("doi", "count"), median_citations=("citations", "median"))
         .query("articles >= 20")
         .sort_values("median_citations", ascending=False).head(15))
top

## 8. Field mix over time

In [ ]:
(df.dropna(subset=["field", "published_year"])
   .groupby(["published_year", "field"]).size()
   .unstack(fill_value=0).plot.area(figsize=(12, 6), cmap="tab20"))

## Next steps

See `KAGGLE.md` for analysis ideas, caveats and provenance.